In [1]:
import numpy as np
import pandas as pd 
from dateutil import parser

In [2]:
df = pd.read_csv("healthcare_messy_data.csv")

In [3]:
df.head(5)

,Patient Name,Age,Gender,Condition,Medication,Visit Date,Blood Pressure,Cholesterol,Email,Phone Number
0,david lee,25,Other,Heart Disease,METFORMIN,01/15/2020,140/90,200.0,name@hospital.org,555-555-5555
1,emily davis,NaN,Male,Diabetes,NONE,"April 5, 2018",120/80,200.0,NaN,NaN
2,laura martinez,35,Other,Asthma,METFORMIN,2019.12.01,110/70,160.0,contact@domain.com,NaN
3,michael wilson,NaN,Male,Diabetes,ALBUTEROL,01/15/2020,110/70,NaN,name@hospital.org,555-555-5555
4,david lee,NaN,Female,Asthma,NONE,2020/02/20,110/70,180.0,NaN,


In [4]:
print('PRELIMINARY AUDIT')
sep = '*' * 40
space = ' ' * 50

print(sep)
print(f'This is the shape of our dataset {df.shape}')
print(space)
print(space)


print(sep)
print(f'High level overview of our column names and datatype \n {df.dtypes}')
print(space)
print(space)


print(sep)
print(f'This is a row level overview of our data set, observing for missing values and other anomalies')
df.isna().sum().reset_index()

PRELIMINARY AUDIT
****************************************
This is the shape of our dataset (1000, 10)
                                                  
                                                  
****************************************
High level overview of our column names and datatype 
 Patient Name       object
Age                object
Gender             object
Condition          object
Medication         object
Visit Date         object
Blood Pressure     object
Cholesterol       float64
Email              object
Phone Number       object
dtype: object
                                                  
                                                  
****************************************
This is a row level overview of our data set, observing for missing values and other anomalies


,index,0
0,Patient Name,0
1,Age,159
2,Gender,0
3,Condition,206
4,Medication,0
5,Visit Date,0
6,Blood Pressure,166
7,Cholesterol,231
8,Email,384
9,Phone Number,179


Our initial examiniation shows inconsistencies such as wrong data type, missing values, trailing spaces and others. We would therefore engage in cleaning and standardization of our dataset.

In [5]:
#Standardizing column names.
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
df.head()

,patient_name,age,gender,condition,medication,visit_date,blood_pressure,cholesterol,email,phone_number
0,david lee,25,Other,Heart Disease,METFORMIN,01/15/2020,140/90,200.0,name@hospital.org,555-555-5555
1,emily davis,NaN,Male,Diabetes,NONE,"April 5, 2018",120/80,200.0,NaN,NaN
2,laura martinez,35,Other,Asthma,METFORMIN,2019.12.01,110/70,160.0,contact@domain.com,NaN
3,michael wilson,NaN,Male,Diabetes,ALBUTEROL,01/15/2020,110/70,NaN,name@hospital.org,555-555-5555
4,david lee,NaN,Female,Asthma,NONE,2020/02/20,110/70,180.0,NaN,


In [6]:
#Select all Object data types.
obj_cols = list(df.dtypes[df.dtypes == 'object'].index)
obj_cols

['patient_name',
 'age',
 'gender',
 'condition',
 'medication',
 'visit_date',
 'blood_pressure',
 'email',
 'phone_number']

In [7]:
#Loop over the object index list and apply capitlization, remove trailing spaces and also replace spaces with underscore
for col in obj_cols:
    df[col] = df[col].str.strip().str.lower()

df.head()

,patient_name,age,gender,condition,medication,visit_date,blood_pressure,cholesterol,email,phone_number
0,david lee,25,other,heart disease,metformin,01/15/2020,140/90,200.0,name@hospital.org,555-555-5555
1,emily davis,NaN,male,diabetes,none,"april 5, 2018",120/80,200.0,NaN,NaN
2,laura martinez,35,other,asthma,metformin,2019.12.01,110/70,160.0,contact@domain.com,NaN
3,michael wilson,NaN,male,diabetes,albuterol,01/15/2020,110/70,NaN,name@hospital.org,555-555-5555
4,david lee,NaN,female,asthma,none,2020/02/20,110/70,180.0,NaN,


Now we start treating the inconsistencies in the columns specifically

In [8]:
#We cannot use unique values to get an idea of what the column contains because different rows can have the save value, so we use the value_counts().
df['age'].value_counts()

age
forty    176
25       171
60       170
35       163
70       161
Name: count, dtype: int64

In [9]:
#Replacing 'forty' in our age column with actual number 40, and convert the column to its proper data type
df['age'] = df['age'].replace('forty', 40)
df['age'].value_counts()


age
40    176
25    171
60    170
35    163
70    161
Name: count, dtype: int64

In [ ]:
#Convert Age column data type from object to Int
df['age'] = pd.to_numeric(df['age'], errors='coerce').astype('Int64')
df.dtypes['age']


Int64Dtype()

We systematically choose a subset of our dataframe to del with the missing values. Since this is information the hospital have to obtain themselves, we convert them to unknow, because we cannot supply them ourselves. Also, these columns missing values are easy to deal with, so we handle them first.

In [11]:
#We would first and foremost treat phone number specially because some rows are empty spaces
df['phone_number'] = df['phone_number'].replace('', 'Unknown')

In [12]:
df[['email', 'phone_number' ]] = df[['email', 'phone_number']].fillna('Unknown')
df.isna().sum().reset_index()

,index,0
0,patient_name,0
1,age,159
2,gender,0
3,condition,206
4,medication,0
5,visit_date,0
6,blood_pressure,166
7,cholesterol,231
8,email,0
9,phone_number,0


In [13]:
#Dealing with the date column, standardizing into  proper date/time format. 

def parse_flexible_date(val):
    if pd.isna(val) or val in ['nan', 'None', '', 'NaT']:
        return pd.NaT
    try:
         # Clean underscores first
        val_clean = str(val).replace("_", " ")
         # fuzzy=True ignores non-date words/characters
        return parser.parse(val_clean, fuzzy=True)
    except Exception:
        return pd.NaT

df['visit_date'] = df['visit_date'].apply(parse_flexible_date)

In [14]:
df['visit_date']

0     2020-01-15
1     2018-04-05
2     2019-12-01
3     2020-01-15
4     2020-02-20
         ...    
995   2019-03-25
996   2020-01-15
997   2020-02-20
998   2018-04-05
999   2020-02-20
Name: visit_date, Length: 1000, dtype: datetime64[ns]

In [15]:
df.head(20)

,patient_name,age,gender,condition,medication,visit_date,blood_pressure,cholesterol,email,phone_number
0,david lee,25,other,heart disease,metformin,2020-01-15,140/90,200.0,name@hospital.org,555-555-5555
1,emily davis,<NA>,male,diabetes,none,2018-04-05,120/80,200.0,Unknown,Unknown
2,laura martinez,35,other,asthma,metformin,2019-12-01,110/70,160.0,contact@domain.com,Unknown
3,michael wilson,<NA>,male,diabetes,albuterol,2020-01-15,110/70,NaN,name@hospital.org,555-555-5555
4,david lee,<NA>,female,asthma,none,2020-02-20,110/70,180.0,Unknown,Unknown
5,mary clark,<NA>,male,hypertension,metformin,2019-03-25,140/90,180.0,Unknown,Unknown
6,robert brown,<NA>,male,hypertension,lisinopril,2020-02-20,120/80,NaN,name@hospital.org,Unknown
7,david lee,60,other,NaN,none,2020-02-20,NaN,NaN,name@hospital.org,Unknown
8,laura martinez,60,other,heart disease,none,2020-01-15,140/90,180.0,name@hospital.org,Unknown
9,michael wilson,40,female,NaN,albuterol,2020-02-20,140/90,220.0,patient@example.com,555-555-5555


In [16]:
#We first check for duplicate values
df.duplicated().sum()
df.drop_duplicates(inplace=True)

In [17]:
#We want to check for and handle missing data in out dataset
df.isna().sum()

patient_name        0
age               159
gender              0
condition         206
medication          0
visit_date          0
blood_pressure    166
cholesterol       231
email               0
phone_number        0
dtype: int64

We have decided to adopt a framework to drop missing value. 
1. if a patient record (row) is having missing records in its condition, blood pressure and cholesterol.
First we split the blood pressure column into systolic (upper value) and diastolic (lower value) columns.

In [18]:
df[['systolic_bp', 'diastolic_bp']] = df['blood_pressure'].str.split('/', expand=True)

Next we strip for any trailing spaces, anc convert the data types from object to integers

In [19]:
df['systolic_bp'] = df['systolic_bp'].str.strip()
df['diastolic_bp'] = df['diastolic_bp'].str.strip()

In [20]:
df['systolic_bp'] = pd.to_numeric(df['systolic_bp'], errors='coerce')
df['diastolic_bp'] = pd.to_numeric(df['diastolic_bp'], errors='coerce')

In [21]:
df.dtypes

patient_name              object
age                        Int64
gender                    object
condition                 object
medication                object
visit_date        datetime64[ns]
blood_pressure            object
cholesterol              float64
email                     object
phone_number              object
systolic_bp              float64
diastolic_bp             float64
dtype: object

To handle the missing values in the new cholesterol readings, we define a criteria to drop any row that has more than 2 missing values in a set of critical columns which are age, gender, and condition. We have determined aligning with conventional health care practices that imputing values for any of these rows where more columns of these critical rows are missing is basically inventing a new patient. 

In [ ]:
#Defining the critical columns
critical_columns = ['age', 'blood_pressure', 'condition', 'cholesterol']

In [23]:
#Count missing critical values per row
missing_per_row = df[critical_columns].isna().sum(axis=1)

In [24]:
#Check the impact
total_rows = len(df)
rows_to_drop = (missing_per_row >= 3).sum()
percent_lost = (rows_to_drop / total_rows) * 100

In [25]:
check = pd.DataFrame({
    'Metric': ['Total Records', 'Rows to Drop', 'Percentage Impact'],
    'Value': [total_rows, rows_to_drop, f"{percent_lost:.2f}%"]
},)

check

,Metric,Value
0,Total Records,1000
1,Rows to Drop,6
2,Percentage Impact,0.60%


Having established that 6 rows (0.60%) rows meet the criteria we have set, and the impact is 2.10% of our entire dataset, the impact of dropping it is negligible, so we drop the dataset. 

In [26]:
#We keep the rows that are less than 3 missing critical fields
df = df[missing_per_row < 3].copy()

In [27]:
df.head()

,patient_name,age,gender,condition,medication,visit_date,blood_pressure,cholesterol,email,phone_number,systolic_bp,diastolic_bp
0,david lee,25,other,heart disease,metformin,2020-01-15,140/90,200.0,name@hospital.org,555-555-5555,140.0,90.0
1,emily davis,<NA>,male,diabetes,none,2018-04-05,120/80,200.0,Unknown,Unknown,120.0,80.0
2,laura martinez,35,other,asthma,metformin,2019-12-01,110/70,160.0,contact@domain.com,Unknown,110.0,70.0
3,michael wilson,<NA>,male,diabetes,albuterol,2020-01-15,110/70,NaN,name@hospital.org,555-555-5555,110.0,70.0
4,david lee,<NA>,female,asthma,none,2020-02-20,110/70,180.0,Unknown,Unknown,110.0,70.0


In [28]:
#We impute unknown for missing values in our gender column
df['gender'] = df['gender'].fillna('unknown')

In [29]:
#Impute missing values for age using the median age by gender
#Convert it first to float so you can apply the fillna function
df['age'] = df['age'].astype('float64')
df['age'] = df['age'].fillna(df.groupby('gender')['age'].transform('median')).fillna(df['age'].median())

#Convert back to Int
df['age'] = df['age'].round().astype('Int64')

In [30]:
#Create Age brackets
df['age_group'] = pd.cut(
    df['age'],
    bins = [0, 30, 45, 60, 75, 120],
    labels = ['<30', '30-44', '45-59', '60-74', '75+'],
    right=False)

In [31]:
#Tracking Original Missingness
df['cholesterol_imputed'] = df['cholesterol'].isna()
df['systolic_bp_imputed'] = df['systolic_bp'].isna()
df['diastolic_bp_imputed'] = df['diastolic_bp'].isna()


We are going to impute missing values with median imputation and cascading fallback
for systolic we are using the median of age group and gender
For diastolic we are using the median of gender
For cholesterol, we are using a combination of the median of the overall dataset


In [32]:
imputed_cols = ['systolic_bp', 'diastolic_bp', 'cholesterol']

for col in imputed_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    #calculate cascading medians
    subgroup_median = df.groupby(['age_group', 'gender'], observed=False)[col].transform('median')
    gender_median = df.groupby('gender', observed=False)[col].transform('median')
    global_median = df[col].median()

    #Impute missing values with cascading fallback
    df[col] = df[col].fillna(subgroup_median).fillna(gender_median).fillna(global_median)

In [33]:
df[['systolic_bp', 'diastolic_bp']] = df[['systolic_bp', 'diastolic_bp']].round().astype('Int64')

df['cholesterol'] = df['cholesterol'].round().astype('Int64')

In [34]:
# STEP 5: VERIFICATION & AUDIT
# ==============================================================================
print("=== REMAINING MISSING VALUES (Should all be 0) ===")
print(df[imputed_cols].isna().sum())
print("\n=== SUMMARY STATISTICS AFTER IMPUTATION ===")
print(df[imputed_cols].describe())

=== REMAINING MISSING VALUES (Should all be 0) ===
systolic_bp     0
diastolic_bp    0
cholesterol     0
dtype: int64

=== SUMMARY STATISTICS AFTER IMPUTATION ===
       systolic_bp  diastolic_bp  cholesterol
count        994.0         994.0        994.0
mean    125.372233     81.624748   189.416499
std      10.663406      6.995973    20.167016
min          110.0          70.0        160.0
25%          120.0          80.0        180.0
50%          130.0          85.0        180.0
75%          130.0          85.0        200.0
max          140.0          90.0        220.0


In [35]:
df.head(10)

,patient_name,age,gender,condition,medication,visit_date,blood_pressure,cholesterol,email,phone_number,systolic_bp,diastolic_bp,age_group,cholesterol_imputed,systolic_bp_imputed,diastolic_bp_imputed
0,david lee,25,other,heart disease,metformin,2020-01-15,140/90,200,name@hospital.org,555-555-5555,140,90,<30,False,False,False
1,emily davis,40,male,diabetes,none,2018-04-05,120/80,200,Unknown,Unknown,120,80,30-44,False,False,False
2,laura martinez,35,other,asthma,metformin,2019-12-01,110/70,160,contact@domain.com,Unknown,110,70,30-44,False,False,False
3,michael wilson,40,male,diabetes,albuterol,2020-01-15,110/70,180,name@hospital.org,555-555-5555,110,70,30-44,True,False,False
4,david lee,40,female,asthma,none,2020-02-20,110/70,180,Unknown,Unknown,110,70,30-44,False,False,False
5,mary clark,40,male,hypertension,metformin,2019-03-25,140/90,180,Unknown,Unknown,140,90,30-44,False,False,False
6,robert brown,40,male,hypertension,lisinopril,2020-02-20,120/80,180,name@hospital.org,Unknown,120,80,30-44,True,False,False
7,david lee,60,other,NaN,none,2020-02-20,NaN,180,name@hospital.org,Unknown,120,80,60-74,True,True,True
8,laura martinez,60,other,heart disease,none,2020-01-15,140/90,180,name@hospital.org,Unknown,140,90,60-74,False,False,False
9,michael wilson,40,female,NaN,albuterol,2020-02-20,140/90,220,patient@example.com,555-555-5555,140,90,30-44,False,False,False


Dealing with Outliers. We first try to determine outliers in our dataset using various methods:
1. ClinicalLY impossible bounds
2. IQR 


In [36]:
clinical_limits = {
    'age': (0, 115),
    'systolic_bp': (60, 240),
    'diastolic_bp': (40, 140),
    'cholesterol': (70, 500)
}

In [37]:
#Find rows violating physical limits
for col, (low, high) in clinical_limits.items():
    if col in df.columns:
        violations = df[(df[col] < low) | (df[col] > high)]
        print(f"{col}: {len(violations)} imposible values found outside ({low} - {high})")
    else:
        print(f'All is okay with this value')

age: 0 imposible values found outside (0 - 115)
systolic_bp: 0 imposible values found outside (60 - 240)
diastolic_bp: 0 imposible values found outside (40 - 140)
cholesterol: 0 imposible values found outside (70 - 500)


In [38]:
#Using the IQR
def detect_outliers_iqr(df, cols, threshhold=3.0):
    outlier_summary = []

    for col in cols:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1

        lower_bound = q1 - (threshhold * iqr)
        upper_bound = q3 + (threshhold * iqr)

        outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]

        outlier_summary.append({
            'Column' : col,
            'Lower Bound': round(lower_bound, 1),
            'Upper Bound': round(upper_bound, 1),
            'Outlier Count': len(outliers),
            'Outlier %': f'{(len(outliers) / len(df)) * 100:2f}%'
        })

    return pd.DataFrame(outlier_summary)


int_cols = df.select_dtypes(include=['int64', 'Int64', 'float64']).columns
detect_outliers_iqr(df, int_cols)

,Column,Lower Bound,Upper Bound,Outlier Count,Outlier %
0,age,-40.0,135.0,0,0.000000%
1,cholesterol,120.0,260.0,0,0.000000%
2,systolic_bp,90.0,160.0,0,0.000000%
3,diastolic_bp,65.0,100.0,0,0.000000%


In [39]:
df.drop(columns='blood_pressure', inplace=True)
df.head()

,patient_name,age,gender,condition,medication,visit_date,cholesterol,email,phone_number,systolic_bp,diastolic_bp,age_group,cholesterol_imputed,systolic_bp_imputed,diastolic_bp_imputed
0,david lee,25,other,heart disease,metformin,2020-01-15,200,name@hospital.org,555-555-5555,140,90,<30,False,False,False
1,emily davis,40,male,diabetes,none,2018-04-05,200,Unknown,Unknown,120,80,30-44,False,False,False
2,laura martinez,35,other,asthma,metformin,2019-12-01,160,contact@domain.com,Unknown,110,70,30-44,False,False,False
3,michael wilson,40,male,diabetes,albuterol,2020-01-15,180,name@hospital.org,555-555-5555,110,70,30-44,True,False,False
4,david lee,40,female,asthma,none,2020-02-20,180,Unknown,Unknown,110,70,30-44,False,False,False


In [40]:
df['condition'] = df['condition'].fillna('Unknown')
df.isna().sum().reset_index()

,index,0
0,patient_name,0
1,age,0
2,gender,0
3,condition,0
4,medication,0
5,visit_date,0
6,cholesterol,0
7,email,0
8,phone_number,0
9,systolic_bp,0


In [41]:
bool_cols = df.select_dtypes(include='bool').columns

In [42]:
df[bool_cols] = df[bool_cols].replace({True : 'Yes', False: 'No'})
df.head()

,patient_name,age,gender,condition,medication,visit_date,cholesterol,email,phone_number,systolic_bp,diastolic_bp,age_group,cholesterol_imputed,systolic_bp_imputed,diastolic_bp_imputed
0,david lee,25,other,heart disease,metformin,2020-01-15,200,name@hospital.org,555-555-5555,140,90,<30,No,No,No
1,emily davis,40,male,diabetes,none,2018-04-05,200,Unknown,Unknown,120,80,30-44,No,No,No
2,laura martinez,35,other,asthma,metformin,2019-12-01,160,contact@domain.com,Unknown,110,70,30-44,No,No,No
3,michael wilson,40,male,diabetes,albuterol,2020-01-15,180,name@hospital.org,555-555-5555,110,70,30-44,Yes,No,No
4,david lee,40,female,asthma,none,2020-02-20,180,Unknown,Unknown,110,70,30-44,No,No,No


In [ ]:
df.to_csv('cleaned_healthcare_data.csv', index=False)